# Stochastic Gradient Descent (SGD)

**Date:** 2026-04-26

---

## What is SGD?

In **Stochastic Gradient Descent**, we compute the gradient using **one random training sample** at a time and update parameters immediately after each sample.

### Update Rule

$$\theta = \theta - \alpha \cdot \nabla_{\theta} L(x_i, y_i)$$

- $i$ — a single randomly picked sample
- $m$ updates happen per epoch (one per sample)
- "Stochastic" = random — we pick samples randomly

### Why "stochastic"?
Each gradient is computed from a single random sample, so it's a noisy estimate of the true gradient. This noise is actually useful — it helps escape local minima.

### Pros
- Very fast — starts updating immediately, even after 1 sample
- Lower memory — only needs 1 sample at a time
- Noise can help escape local minima and saddle points
- Works for online learning (new data arrives continuously)

### Cons
- High variance — loss curve is very noisy / oscillates
- May never fully converge — keeps bouncing near the minimum
- Cannot leverage GPU vectorization well (batch size = 1)
- Requires careful learning rate tuning (or scheduling)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Generate Data

In [ ]:
m = 1000
X = np.random.randn(m, 1)
y = 3 * X + 2 + np.random.randn(m, 1) * 0.5

plt.scatter(X, y, alpha=0.3, s=10, color='darkorange')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Training Data  (y = 3x + 2 + noise)')
plt.show()

## 2. Helper Functions

In [ ]:
def predict(X, W, b):
    return X @ W + b

def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

def compute_gradients(X, y, W, b):
    n = len(y)
    error = predict(X, W, b) - y
    dW = (2 / n) * X.T @ error
    db = (2 / n) * np.sum(error)
    return dW, db

## 3. SGD — Implementation

Each epoch:
1. Shuffle the dataset
2. For **each sample** — compute gradient on that 1 sample → update weights immediately
3. = **m updates per epoch**

In [ ]:
def stochastic_gradient_descent(X, y, lr=0.01, epochs=50):
    W = np.zeros((X.shape[1], 1))
    b = 0.0
    history = {'loss': [], 'W': [], 'b': [], 'updates_per_epoch': len(y)}

    for epoch in range(epochs):
        indices = np.random.permutation(len(y))  # shuffle
        for i in indices:
            Xi = X[i:i+1]   # single sample, shape (1,1)
            yi = y[i:i+1]   # single label,  shape (1,1)

            dW, db = compute_gradients(Xi, yi, W, b)  # gradient on 1 sample
            W -= lr * dW
            b -= lr * db

        # Evaluate on full dataset at end of epoch
        loss = mse_loss(predict(X, W, b), y)
        history['loss'].append(loss)
        history['W'].append(W[0, 0])
        history['b'].append(b)

    return W, b, history

W_final, b_final, history = stochastic_gradient_descent(X, y, lr=0.01, epochs=50)

print(f"Learned  ->  W = {W_final[0,0]:.4f},  b = {b_final:.4f}")
print(f"True     ->  W = 3.0000,  b = 2.0000")
print(f"Final MSE Loss: {history['loss'][-1]:.4f}")
print(f"Updates per epoch: {history['updates_per_epoch']} (vs 1 in Batch GD)")

## 4. Visualize Results — Notice the Noise!

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['loss'], color='darkorange', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss Curve — SGD  (noisy!)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['W'], color='darkorange', linewidth=1.5)
axes[1].axhline(y=3, color='red', linestyle='--', label='True W=3')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('W')
axes[1].set_title('Weight W Convergence  (oscillates)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['b'], color='darkorange', linewidth=1.5)
axes[2].axhline(y=2, color='red', linestyle='--', label='True b=2')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('b')
axes[2].set_title('Bias b Convergence  (oscillates)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Stochastic Gradient Descent (SGD)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Why is SGD Noisy? — Visualizing a Single Epoch

Let's track the loss **within a single epoch** (after each sample update) to see how much it fluctuates.

In [ ]:
W = np.array([[2.0]])  # start away from optimum
b = 0.0
intra_epoch_losses = []

indices = np.random.permutation(m)
for i in indices[:200]:  # show first 200 updates of one epoch
    Xi = X[i:i+1]
    yi = y[i:i+1]
    dW, db_grad = compute_gradients(Xi, yi, W, b)
    W -= 0.01 * dW
    b -= 0.01 * db_grad
    intra_epoch_losses.append(mse_loss(predict(X, W, b), y))

plt.figure(figsize=(12, 4))
plt.plot(intra_epoch_losses, color='darkorange', linewidth=1, alpha=0.8)
plt.xlabel('Update step (within one epoch)')
plt.ylabel('MSE Loss (full dataset)')
plt.title('SGD: Loss after each single-sample update — within one epoch')
plt.grid(True, alpha=0.3)
plt.show()
print("Notice: loss jumps up and down — each sample pulls weights in a different direction")

## 6. Learning Rate Scheduling

Because SGD is noisy, a common trick is to **decay the learning rate** over time so it takes smaller steps as it nears the minimum.

In [ ]:
def sgd_with_lr_decay(X, y, initial_lr=0.1, decay=0.01, epochs=50):
    W = np.zeros((X.shape[1], 1))
    b = 0.0
    loss_history = []

    for epoch in range(epochs):
        lr = initial_lr / (1 + decay * epoch)  # decaying lr
        indices = np.random.permutation(len(y))
        for i in indices:
            dW, db_g = compute_gradients(X[i:i+1], y[i:i+1], W, b)
            W -= lr * dW
            b -= lr * db_g
        loss_history.append(mse_loss(predict(X, W, b), y))

    return W, b, loss_history

_, _, loss_fixed   = stochastic_gradient_descent(X, y, lr=0.01,  epochs=50)
_, _, loss_decayed = sgd_with_lr_decay(X, y, initial_lr=0.1, decay=0.05, epochs=50)

plt.figure(figsize=(12, 4))
plt.plot(loss_fixed,   label='SGD fixed lr=0.01',         color='darkorange', linewidth=2)
plt.plot(loss_decayed, label='SGD decaying lr (0.1→...)', color='green',      linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('SGD: Fixed vs Decaying Learning Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Key Observations

- **Noisy loss curve** — because each gradient is computed from just 1 sample (high variance)
- **Many updates per epoch** — m=1000 updates vs 1 update in Batch GD; learns faster at start
- **Can escape local minima** — the noise acts as implicit regularization
- **Doesn't fully converge** — keeps oscillating around the minimum; needs learning rate decay
- **Not GPU-friendly** — batch size of 1 means no parallelism

---

**Next:** `03_mini_batch_gradient_descent.ipynb` — the best of both worlds